In [ ]:
import uproot
import pandas as pd
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt

# Read NuSyst Tree ROOT file

In [ ]:
with uproot.open("NuSystTree.root") as f:

    # Note that this time we are using awkward array
    df_event = f["events"].arrays(library="ak")

    df_meta = f["tweak_metadata"].arrays(library="pd")

Printing metadata tree

In [ ]:
df_meta

Now we have reweight values differ from 1.0! Note: only CCQE events will affected by this dial, so non-CCQE events still get reweights of 1.0

In [ ]:
DialColumnName_prefix = "DUNEDAS2026ExampleReweighter_NuSystTutorial"

DialARWColName = f"tweak_responses_{DialColumnName_prefix}_DialA"

df_event[DialARWColName].show(10)

We want to take the "+1 sigma" reweight and apply it to the simulation.
From the metadata, we know that the third (index=2) of each reweight array corresponds to +1 sigma.
This can be done by using python slicing.

In [ ]:
# [:,2]
# - ":" means we are selecting all elements from the "row" (or the first dimension/axis)
# - "2" means we are only selecting the third (index=2) element from the "column" (or the second dimension/axis) 
arr_DialA_pOneSig = df_event[DialARWColName][:,2]
arr_DialA_pOneSig.show(10)

# Now let's draw Q2 distributions
- h: CV
- h_RW: Reweighted using arr_DialA_pOneSig

## First, select CCQE events

Use `is_cc` and `is_qe` variables to obtain a boolean array for CCQE event selection

In [ ]:
IsCCQE = (df_event['is_cc']) & (df_event['is_qe'])
IsCCQE.show(10)

In [ ]:
fig, ax = plt.subplots(figsize=(6,6))

Binning = np.linspace(0, 2.0, 10+1)

h_ccqe = np.histogram(
    df_event[IsCCQE]['Q2'],
    bins=Binning,
)[0]
ax.hist(
    Binning[:-1],
    bins=Binning,
    weights=h_ccqe,
    histtype='step',
    label='MC production (CV)'
)
ax.set_xlim(Binning[0], Binning[-1])
ax.set_xlabel(r'$Q^{2}$ $(GeV^{2})$', fontsize=20)

h_RW_ccqe = np.histogram(
    df_event[IsCCQE]['Q2'],
    bins=Binning,
    weights=arr_DialA_pOneSig[IsCCQE], # NOTE: the reweight array should be also selected
)[0]
ax.hist(
    Binning[:-1],
    bins=Binning,
    weights=h_RW_ccqe,
    histtype='step',
    label='Alternative model'
)

fig.suptitle('CCQE events')
ax.legend(loc='upper right', fontsize=10)


We can also draw the ratio

In [ ]:
fig, ax = plt.subplots(figsize=(6,6))

h_ratio_ccqe = h_RW_ccqe/h_ccqe

ax.hist(
    Binning[:-1],
    bins=Binning,
    weights=h_ratio_ccqe,
    histtype='step',
    label='MC production (CV)'
)
ax.set_ylabel('Ratio to CV', fontsize=20)
ax.set_xlim(Binning[0], Binning[-1])
ax.set_xlabel(r'$Q^{2}$ $(GeV^{2})$', fontsize=20)

fig.suptitle('CCQE events')

## Draw non-CCQE events to verify that they are not affected by this dial

We can flip True-False by applying `~` on `IsCCQE`

In [ ]:
~IsCCQE

In [ ]:
fig, ax = plt.subplots(figsize=(6,6))

Binning = np.linspace(0, 2.0, 10+1)

h_nonccqe = np.histogram(
    df_event[~IsCCQE]['Q2'],
    bins=Binning
)[0]
ax.hist(
    Binning[:-1],
    bins=Binning,
    weights=h_nonccqe,
    histtype='step',
    label='MC production (CV)'
)
ax.set_xlim(Binning[0], Binning[-1])
ax.set_xlabel(r'$Q^{2}$ $(GeV^{2})$', fontsize=20)

h_RW_nonccqe = np.histogram(
    df_event[~IsCCQE]['Q2'],
    bins=Binning,
    weights=arr_DialA_pOneSig[~IsCCQE],
)[0]
ax.hist(
    Binning[:-1],
    bins=Binning,
    weights=h_RW_nonccqe,
    histtype='step',
    label='Alternative model'
)

fig.suptitle('Non CCQE events')
ax.legend(loc='upper right', fontsize=10)


In [ ]:
fig, ax = plt.subplots(figsize=(6,6))

h_ratio_nonccqe = h_RW_nonccqe/h_nonccqe

ax.hist(
    Binning[:-1],
    bins=Binning,
    weights=h_ratio_nonccqe,
    histtype='step',
    label='MC production (CV)'
)
ax.set_ylabel('Ratio to CV', fontsize=20)
ax.set_xlim(Binning[0], Binning[-1])
ax.set_xlabel(r'$Q^{2}$ $(GeV^{2})$', fontsize=20)

fig.suptitle('Non-CCQE events')